# 02 — Data Quality Assessment

## Healthcare Claims & Revenue Analytics

This notebook performs a detailed assessment of data quality issues identified during the initial data validation stage.

### Objectives

- Assess missing-value patterns
- Investigate duplicate and repeated records
- Evaluate data types and coded fields
- Examine financial-value anomalies
- Assess field usability for analysis
- Define data-cleaning requirements
- Document decisions before modifying any data

### Data Quality Dimensions

The assessment will consider:

1. Completeness
2. Uniqueness
3. Validity
4. Consistency
5. Accuracy-related checks
6. Analytical relevance

> **Important:** Raw source data will not be modified during this notebook.

In [1]:
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Project paths
# --------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "00_Working_Data"

print("Project root:")
print(PROJECT_ROOT)

print("\nWorking data directory:")
print(DATA_DIR)


# --------------------------------------------------
# Dataset references
# --------------------------------------------------

files = {
    "beneficiary_2008": DATA_DIR / "DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv",
    "inpatient": DATA_DIR / "DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv",
    "outpatient": DATA_DIR / "DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv",
    "beneficiary_2009": DATA_DIR / "DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv",
    "beneficiary_2010": DATA_DIR / "DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv",
}

print("\nDataset files:")
for name, file_path in files.items():
    print(f"{name:20} | {file_path.name}")

Project root:
c:\Users\acharya\Desktop\Healthcare_Claims_Revenue_Analytics

Working data directory:
c:\Users\acharya\Desktop\Healthcare_Claims_Revenue_Analytics\00_Working_Data

Dataset files:
beneficiary_2008     | DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv
inpatient            | DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv
outpatient           | DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv
beneficiary_2009     | DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv
beneficiary_2010     | DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv


## 1. Completeness Assessment

Completeness measures how much usable information is available in each field.

The assessment will identify:
- Missing-value counts
- Missing-value percentages
- Fields with substantial missingness
- Fields that are completely missing
- Potential candidates for later cleaning or exclusion

In [2]:
# --------------------------------------------------
# Completeness Assessment
# --------------------------------------------------

missingness_results = {}

for name, file_path in files.items():
    print(f"\nAssessing: {name}")
    
    missing_counts = {}
    
    # Process large files in chunks to reduce memory usage
    for chunk in pd.read_csv(
        file_path,
        chunksize=100_000,
        low_memory=False
    ):
        chunk_missing = chunk.isna().sum()
        
        for column, count in chunk_missing.items():
            missing_counts[column] = missing_counts.get(column, 0) + count
    
    # Calculate total records
    total_records = sum(
        1 for _ in open(file_path, encoding="utf-8")
    ) - 1
    
    # Create summary
    missing_df = pd.DataFrame({
        "Missing_Count": missing_counts
    })
    
    missing_df["Total_Records"] = total_records
    
    missing_df["Missing_Percentage"] = (
        missing_df["Missing_Count"]
        / missing_df["Total_Records"]
        * 100
    )
    
    missing_df["Dataset"] = name
    
    missingness_results[name] = missing_df
    
    display(
        missing_df
        .sort_values("Missing_Percentage", ascending=False)
        .head(20)
    )


Assessing: beneficiary_2008


,Missing_Count,Total_Records,Missing_Percentage,Dataset
BENE_DEATH_DT,114538,116352,98.440938,beneficiary_2008
DESYNPUF_ID,0,116352,0.000000,beneficiary_2008
BENE_BIRTH_DT,0,116352,0.000000,beneficiary_2008
BENE_SEX_IDENT_CD,0,116352,0.000000,beneficiary_2008
BENE_RACE_CD,0,116352,0.000000,beneficiary_2008
BENE_ESRD_IND,0,116352,0.000000,beneficiary_2008
SP_STATE_CODE,0,116352,0.000000,beneficiary_2008
BENE_COUNTY_CD,0,116352,0.000000,beneficiary_2008
BENE_HI_CVRAGE_TOT_MONS,0,116352,0.000000,beneficiary_2008
BENE_SMI_CVRAGE_TOT_MONS,0,116352,0.000000,beneficiary_2008



Assessing: inpatient


,Missing_Count,Total_Records,Missing_Percentage,Dataset
HCPCS_CD_29,66773,66773,100.0,inpatient
HCPCS_CD_30,66773,66773,100.0,inpatient
HCPCS_CD_31,66773,66773,100.0,inpatient
HCPCS_CD_32,66773,66773,100.0,inpatient
HCPCS_CD_1,66773,66773,100.0,inpatient
HCPCS_CD_5,66773,66773,100.0,inpatient
HCPCS_CD_4,66773,66773,100.0,inpatient
HCPCS_CD_3,66773,66773,100.0,inpatient
HCPCS_CD_2,66773,66773,100.0,inpatient
HCPCS_CD_10,66773,66773,100.0,inpatient



Assessing: outpatient


,Missing_Count,Total_Records,Missing_Percentage,Dataset
HCPCS_CD_45,790790,790790,100.000000,outpatient
ICD9_PRCDR_CD_6,790761,790790,99.996333,outpatient
ICD9_PRCDR_CD_5,790755,790790,99.995574,outpatient
ICD9_PRCDR_CD_4,790742,790790,99.993930,outpatient
ICD9_PRCDR_CD_3,790717,790790,99.990769,outpatient
ICD9_PRCDR_CD_2,790667,790790,99.984446,outpatient
ICD9_PRCDR_CD_1,790590,790790,99.974709,outpatient
ICD9_DGNS_CD_10,789165,790790,99.794509,outpatient
HCPCS_CD_44,780234,790790,98.665132,outpatient
HCPCS_CD_43,779594,790790,98.584201,outpatient



Assessing: beneficiary_2009


,Missing_Count,Total_Records,Missing_Percentage,Dataset
BENE_DEATH_DT,112754,114538,98.442438,beneficiary_2009
DESYNPUF_ID,0,114538,0.000000,beneficiary_2009
BENE_BIRTH_DT,0,114538,0.000000,beneficiary_2009
BENE_SEX_IDENT_CD,0,114538,0.000000,beneficiary_2009
BENE_RACE_CD,0,114538,0.000000,beneficiary_2009
BENE_ESRD_IND,0,114538,0.000000,beneficiary_2009
SP_STATE_CODE,0,114538,0.000000,beneficiary_2009
BENE_COUNTY_CD,0,114538,0.000000,beneficiary_2009
BENE_HI_CVRAGE_TOT_MONS,0,114538,0.000000,beneficiary_2009
BENE_SMI_CVRAGE_TOT_MONS,0,114538,0.000000,beneficiary_2009



Assessing: beneficiary_2010


,Missing_Count,Total_Records,Missing_Percentage,Dataset
BENE_DEATH_DT,110891,112754,98.34773,beneficiary_2010
DESYNPUF_ID,0,112754,0.00000,beneficiary_2010
BENE_BIRTH_DT,0,112754,0.00000,beneficiary_2010
BENE_SEX_IDENT_CD,0,112754,0.00000,beneficiary_2010
BENE_RACE_CD,0,112754,0.00000,beneficiary_2010
BENE_ESRD_IND,0,112754,0.00000,beneficiary_2010
SP_STATE_CODE,0,112754,0.00000,beneficiary_2010
BENE_COUNTY_CD,0,112754,0.00000,beneficiary_2010
BENE_HI_CVRAGE_TOT_MONS,0,112754,0.00000,beneficiary_2010
BENE_SMI_CVRAGE_TOT_MONS,0,112754,0.00000,beneficiary_2010


In [3]:
# --------------------------------------------------
# Consolidated Missingness Summary
# --------------------------------------------------

all_missingness = pd.concat(
    missingness_results.values(),
    ignore_index=False
)

# Reset index so column names become an explicit column
all_missingness = all_missingness.reset_index()
all_missingness = all_missingness.rename(columns={"index": "Column"})

# Add missingness category
def classify_missingness(pct):
    if pct == 0:
        return "Complete"
    elif pct < 5:
        return "Low (<5%)"
    elif pct < 20:
        return "Moderate (5-20%)"
    elif pct < 50:
        return "High (20-50%)"
    elif pct < 100:
        return "Very High (50-<100%)"
    else:
        return "Completely Missing (100%)"

all_missingness["Missingness_Category"] = (
    all_missingness["Missing_Percentage"]
    .apply(classify_missingness)
)

# Display summary
display(
    all_missingness[
        [
            "Dataset",
            "Column",
            "Missing_Count",
            "Total_Records",
            "Missing_Percentage",
            "Missingness_Category"
        ]
    ]
    .sort_values(
        ["Dataset", "Missing_Percentage"],
        ascending=[True, False]
    )
)

,Dataset,Column,Missing_Count,Total_Records,Missing_Percentage,Missingness_Category
2,beneficiary_2008,BENE_DEATH_DT,114538,116352,98.440938,Very High (50-<100%)
0,beneficiary_2008,DESYNPUF_ID,0,116352,0.000000,Complete
1,beneficiary_2008,BENE_BIRTH_DT,0,116352,0.000000,Complete
3,beneficiary_2008,BENE_SEX_IDENT_CD,0,116352,0.000000,Complete
4,beneficiary_2008,BENE_RACE_CD,0,116352,0.000000,Complete
...,...,...,...,...,...,...
119,outpatient,CLM_PMT_AMT,0,790790,0.000000,Complete
120,outpatient,NCH_PRMRY_PYR_CLM_PD_AMT,0,790790,0.000000,Complete
124,outpatient,NCH_BENE_BLOOD_DDCTBL_LBLTY_AM,0,790790,0.000000,Complete
141,outpatient,NCH_BENE_PTB_DDCTBL_AMT,0,790790,0.000000,Complete


In [4]:
# --------------------------------------------------
# Missingness Category Summary
# --------------------------------------------------

missingness_category_summary = (
    all_missingness
    .groupby(
        ["Dataset", "Missingness_Category"]
    )
    .size()
    .reset_index(name="Field_Count")
)

display(
    missingness_category_summary
    .sort_values(
        ["Dataset", "Field_Count"],
        ascending=[True, False]
    )
)

,Dataset,Missingness_Category,Field_Count
0,beneficiary_2008,Complete,31
1,beneficiary_2008,Very High (50-<100%),1
2,beneficiary_2009,Complete,31
3,beneficiary_2009,Very High (50-<100%),1
4,beneficiary_2010,Complete,31
5,beneficiary_2010,Very High (50-<100%),1
7,inpatient,Completely Missing (100%),45
6,inpatient,Complete,12
9,inpatient,Low (<5%),10
11,inpatient,Very High (50-<100%),7


In [5]:
# --------------------------------------------------
# Completely Missing Fields
# --------------------------------------------------

completely_missing = (
    all_missingness[
        all_missingness["Missing_Percentage"] == 100
    ]
    [
        [
            "Dataset",
            "Column",
            "Missing_Count",
            "Total_Records"
        ]
    ]
    .sort_values(["Dataset", "Column"])
)

display(completely_missing)

,Dataset,Column,Missing_Count,Total_Records
68,inpatient,HCPCS_CD_1,66773,66773
77,inpatient,HCPCS_CD_10,66773,66773
78,inpatient,HCPCS_CD_11,66773,66773
79,inpatient,HCPCS_CD_12,66773,66773
80,inpatient,HCPCS_CD_13,66773,66773
81,inpatient,HCPCS_CD_14,66773,66773
82,inpatient,HCPCS_CD_15,66773,66773
83,inpatient,HCPCS_CD_16,66773,66773
84,inpatient,HCPCS_CD_17,66773,66773
85,inpatient,HCPCS_CD_18,66773,66773


## Completeness Findings

The completeness assessment identified several important patterns:

### Beneficiary Data

The annual beneficiary files contain 32 fields each.

- 31 fields are complete.
- `BENE_DEATH_DT` has approximately 98.3–98.4% missing values across 2008–2010.
- The high missingness of `BENE_DEATH_DT` will be treated as a data characteristic rather than automatically imputed.
- No missingness was identified in the beneficiary identifier or core demographic fields.

### Inpatient Claims

The inpatient dataset contains 81 fields.

- 45 HCPCS fields (`HCPCS_CD_1` through `HCPCS_CD_45`) are completely missing.
- Several other fields have partial missingness.
- The completely missing HCPCS fields appear to represent fields that are not populated in this inpatient extract.
- These fields will be evaluated for analytical relevance during the cleaning stage rather than removed during quality assessment.

### Outpatient Claims

The outpatient dataset contains 76 fields.

- `HCPCS_CD_45` is completely missing.
- Several ICD-9 procedure and later HCPCS fields have very high missingness.
- Core financial fields such as `CLM_PMT_AMT` and `NCH_PRMRY_PYR_CLM_PD_AMT` are complete.
- Missingness will be assessed in relation to the intended analytical use of each field.

### Overall Assessment

Missing values are concentrated in specific fields rather than being uniformly distributed across the datasets.

High missingness alone will not be treated as evidence of invalid data. Cleaning decisions will consider:

1. Field purpose
2. Analytical relevance
3. Whether the field is structurally unavailable
4. Whether missing values carry meaningful information
5. Impact on downstream SQL and Power BI analysis

> **Raw data remains unchanged.**

## 2. Uniqueness and Duplicate Assessment

This assessment evaluates whether records contain duplicate business keys.

### Identifier Rules

- Beneficiary records are identified by `DESYNPUF_ID`.
- Claims contain `CLM_ID` and `SEGMENT`.
- Because `CLM_ID` can occur across multiple claim segments, `CLM_ID + SEGMENT` will be evaluated as the claim-level uniqueness key.

The assessment will distinguish between:

- Exact duplicate rows
- Repeated identifier values
- Duplicate business keys

> Repeated `CLM_ID` values alone will not automatically be treated as duplicate claims.

In [6]:
# --------------------------------------------------
# Uniqueness Assessment
# --------------------------------------------------

def assess_uniqueness(file_path, key_columns):
    total_rows = 0
    unique_keys = set()
    duplicate_key_count = 0

    for chunk in pd.read_csv(
        file_path,
        chunksize=100_000,
        low_memory=False
    ):
        total_rows += len(chunk)

        # Create a composite key
        keys = list(
            zip(*(chunk[column] for column in key_columns))
        )

        unique_keys.update(keys)

    unique_count = len(unique_keys)
    duplicate_rows = total_rows - unique_count

    return {
        "Total_Rows": total_rows,
        "Unique_Keys": unique_count,
        "Duplicate_Key_Rows": duplicate_rows
    }


uniqueness_results = []

# Beneficiary uniqueness
for name in [
    "beneficiary_2008",
    "beneficiary_2009",
    "beneficiary_2010"
]:
    result = assess_uniqueness(
        files[name],
        ["DESYNPUF_ID"]
    )

    result["Dataset"] = name
    result["Key"] = "DESYNPUF_ID"

    uniqueness_results.append(result)


# Claims uniqueness
for name in ["inpatient", "outpatient"]:
    result = assess_uniqueness(
        files[name],
        ["CLM_ID", "SEGMENT"]
    )

    result["Dataset"] = name
    result["Key"] = "CLM_ID + SEGMENT"

    uniqueness_results.append(result)


uniqueness_df = pd.DataFrame(uniqueness_results)

uniqueness_df = uniqueness_df[
    [
        "Dataset",
        "Key",
        "Total_Rows",
        "Unique_Keys",
        "Duplicate_Key_Rows"
    ]
]

display(uniqueness_df)

,Dataset,Key,Total_Rows,Unique_Keys,Duplicate_Key_Rows
0,beneficiary_2008,DESYNPUF_ID,116352,116352,0
1,beneficiary_2009,DESYNPUF_ID,114538,114538,0
2,beneficiary_2010,DESYNPUF_ID,112754,112754,0
3,inpatient,CLM_ID + SEGMENT,66773,66773,0
4,outpatient,CLM_ID + SEGMENT,790790,790790,0


## Uniqueness Findings

The uniqueness assessment found no duplicate business keys across the datasets.

### Beneficiary Data

The `DESYNPUF_ID` identifier is unique within each annual beneficiary file:

- 2008: 116,352 unique identifiers
- 2009: 114,538 unique identifiers
- 2010: 112,754 unique identifiers

No duplicate beneficiary identifiers were identified within any annual file.

### Inpatient Claims

The combination of `CLM_ID + SEGMENT` is unique across all 66,773 inpatient records.

Therefore, repeated `CLM_ID` values observed during validation do not represent duplicate claim records when the claim segment is considered.

### Outpatient Claims

The combination of `CLM_ID + SEGMENT` is unique across all 790,790 outpatient records.

Therefore, repeated `CLM_ID` values do not represent duplicate business keys in the outpatient dataset.

### Cleaning Decision

No records will be removed based on business-key duplication.

The `CLM_ID + SEGMENT` combination will be treated as the claim-level uniqueness key for downstream analysis.

> **Important:** Duplicate identifiers and exact duplicate rows are different concepts. Exact-row duplication will be assessed separately before cleaning.

## 3. Exact Duplicate Row Assessment

Business-key uniqueness does not necessarily rule out exact duplicate rows.

This assessment checks whether complete records are repeated across each dataset.

### Assessment Approach

- Compare complete rows rather than individual identifiers.
- Process claims data in chunks to avoid loading all large datasets into memory unnecessarily.
- No records will be removed during this assessment.

An exact duplicate is defined as a record where all available field values are identical to another record in the same dataset.

In [7]:
# --------------------------------------------------
# Exact Duplicate Row Assessment
# --------------------------------------------------

def count_exact_duplicates(file_path):
    total_rows = 0
    duplicate_rows = 0
    seen_rows = set()

    for chunk in pd.read_csv(
        file_path,
        chunksize=100_000,
        low_memory=False
    ):
        total_rows += len(chunk)

        # Convert each complete row into a hashable tuple
        for row in chunk.itertuples(index=False, name=None):
            if row in seen_rows:
                duplicate_rows += 1
            else:
                seen_rows.add(row)

    return {
        "Total_Rows": total_rows,
        "Exact_Duplicate_Rows": duplicate_rows
    }


exact_duplicate_results = []

for name, file_path in files.items():

    print(f"Assessing exact duplicates: {name}")

    result = count_exact_duplicates(file_path)

    result["Dataset"] = name

    exact_duplicate_results.append(result)


exact_duplicates_df = pd.DataFrame(exact_duplicate_results)

exact_duplicates_df = exact_duplicates_df[
    [
        "Dataset",
        "Total_Rows",
        "Exact_Duplicate_Rows"
    ]
]

display(exact_duplicates_df)

Assessing exact duplicates: beneficiary_2008
Assessing exact duplicates: inpatient
Assessing exact duplicates: outpatient
Assessing exact duplicates: beneficiary_2009
Assessing exact duplicates: beneficiary_2010


,Dataset,Total_Rows,Exact_Duplicate_Rows
0,beneficiary_2008,116352,0
1,inpatient,66773,0
2,outpatient,790790,0
3,beneficiary_2009,114538,0
4,beneficiary_2010,112754,0


## Exact Duplicate Findings

The exact duplicate assessment found no completely duplicated records in any dataset.

### Results

- Beneficiary 2008: 0 exact duplicate rows
- Beneficiary 2009: 0 exact duplicate rows
- Beneficiary 2010: 0 exact duplicate rows
- Inpatient claims: 0 exact duplicate rows
- Outpatient claims: 0 exact duplicate rows

### Overall Assessment

The datasets do not contain exact duplicate rows based on comparison of all available fields.

Combined with the business-key assessment, this indicates that:

- `DESYNPUF_ID` is unique within each annual beneficiary file.
- `CLM_ID + SEGMENT` is unique within both claims datasets.
- No complete duplicate records were identified.

### Cleaning Decision

No records will be removed because of exact duplication.

Duplicate handling is therefore not expected to be a major data-cleaning requirement for this project.

> **Important:** This conclusion applies to exact duplicates and the defined business keys. It does not imply that every value in every field is correct or clinically accurate.

## 4. Coded Field Validity Assessment

Healthcare claims datasets contain many coded fields.

This assessment reviews selected categorical and indicator fields for:

- Distinct values
- Unexpected values
- Potential placeholder values
- Values outside expected coding patterns

The objective is to identify validity issues before defining cleaning rules.

> **Important:** Observed values will not be changed during this assessment.

In [8]:
# --------------------------------------------------
# Coded Field Validity Assessment
# --------------------------------------------------

beneficiary_fields = [
    "BENE_SEX_IDENT_CD",
    "BENE_RACE_CD",
    "BENE_ESRD_IND",
    "SP_ALZHDMTA",
    "SP_CHF",
    "SP_CHRNKIDN",
    "SP_CNCR",
    "SP_COPD",
    "SP_DEPRESSN",
    "SP_DIABETES",
    "SP_ISCHMCHT",
    "SP_OSTEOPRS",
    "SP_RA_OA",
    "SP_STRKETIA"
]

for name in [
    "beneficiary_2008",
    "beneficiary_2009",
    "beneficiary_2010"
]:

    print(f"\n{'=' * 60}")
    print(f"{name}")
    print(f"{'=' * 60}")

    df_sample = pd.read_csv(
        files[name],
        usecols=beneficiary_fields,
        low_memory=False
    )

    for column in beneficiary_fields:
        values = sorted(
            df_sample[column]
            .dropna()
            .unique()
            .tolist()
        )

        print(f"{column:20} | {values}")


beneficiary_2008
BENE_SEX_IDENT_CD    | [1, 2]
BENE_RACE_CD         | [1, 2, 3, 5]
BENE_ESRD_IND        | ['0', 'Y']
SP_ALZHDMTA          | [1, 2]
SP_CHF               | [1, 2]
SP_CHRNKIDN          | [1, 2]
SP_CNCR              | [1, 2]
SP_COPD              | [1, 2]
SP_DEPRESSN          | [1, 2]
SP_DIABETES          | [1, 2]
SP_ISCHMCHT          | [1, 2]
SP_OSTEOPRS          | [1, 2]
SP_RA_OA             | [1, 2]
SP_STRKETIA          | [1, 2]

beneficiary_2009
BENE_SEX_IDENT_CD    | [1, 2]
BENE_RACE_CD         | [1, 2, 3, 5]
BENE_ESRD_IND        | ['0', 'Y']
SP_ALZHDMTA          | [1, 2]
SP_CHF               | [1, 2]
SP_CHRNKIDN          | [1, 2]
SP_CNCR              | [1, 2]
SP_COPD              | [1, 2]
SP_DEPRESSN          | [1, 2]
SP_DIABETES          | [1, 2]
SP_ISCHMCHT          | [1, 2]
SP_OSTEOPRS          | [1, 2]
SP_RA_OA             | [1, 2]
SP_STRKETIA          | [1, 2]

beneficiary_2010
BENE_SEX_IDENT_CD    | [1, 2]
BENE_RACE_CD         | [1, 2, 3, 5]
BENE_ESRD_IND       

## Coded Field Validity Findings

The selected beneficiary coded fields show consistent value patterns across the 2008, 2009, and 2010 files.

### Demographic Codes

- `BENE_SEX_IDENT_CD` contains only values `1` and `2`.
- `BENE_RACE_CD` contains values `1`, `2`, `3`, and `5`.

### ESRD Indicator

- `BENE_ESRD_IND` contains only `0` and `Y`.

### Chronic Condition Indicators

The selected chronic-condition fields contain only values `1` and `2`:

- `SP_ALZHDMTA`
- `SP_CHF`
- `SP_CHRNKIDN`
- `SP_CNCR`
- `SP_COPD`
- `SP_DEPRESSN`
- `SP_DIABETES`
- `SP_ISCHMCHT`
- `SP_OSTEOPRS`
- `SP_RA_OA`
- `SP_STRKETIA`

### Assessment

No unexpected values were identified in the selected coded fields.

The same coding patterns are present across all three annual beneficiary datasets, indicating good structural consistency.

### Cleaning Decision

The observed coded values will be retained during the quality-assessment stage.

Code-to-label transformations will be performed only after confirming the official DE-SynPUF code definitions.

> **Important:** Numeric codes such as `1`, `2`, `3`, and `5` are categorical codes, not continuous numerical measurements.

## 5. Claims Coded Field Validity Assessment

The claims datasets contain several coded fields used to describe claim structure,
diagnoses, procedures, and services.

This assessment focuses on:

- `SEGMENT`
- ICD-9 diagnosis codes
- ICD-9 procedure codes
- HCPCS codes

The objective is to identify unexpected values, unusual coding patterns,
and fields that require special handling during cleaning.

> **Important:** Code values will not be modified during this assessment.

In [9]:
# --------------------------------------------------
# Claims Coded Field Validity Assessment
# --------------------------------------------------

def inspect_claim_codes(file_path, dataset_name):
    print(f"\n{'=' * 70}")
    print(dataset_name)
    print(f"{'=' * 70}")

    # Read only the structural and coded fields needed for assessment
    df = pd.read_csv(
        file_path,
        usecols=lambda column: (
            column == "SEGMENT"
            or column.startswith("ICD9_DGNS_CD_")
            or column.startswith("ICD9_PRCDR_CD_")
            or column.startswith("HCPCS_CD_")
        ),
        low_memory=False
    )

    # SEGMENT values
    print("\nSEGMENT values:")
    print(sorted(df["SEGMENT"].dropna().unique().tolist()))

    # ICD-9 diagnosis fields
    diagnosis_columns = [
        column for column in df.columns
        if column.startswith("ICD9_DGNS_CD_")
    ]

    print("\nICD-9 diagnosis code patterns:")
    for column in diagnosis_columns:
        non_missing = df[column].dropna()

        print(
            f"{column:20} | "
            f"Non-missing: {len(non_missing):,} | "
            f"Unique: {non_missing.nunique():,} | "
            f"Sample: {non_missing.astype(str).head(5).tolist()}"
        )

    # ICD-9 procedure fields
    procedure_columns = [
        column for column in df.columns
        if column.startswith("ICD9_PRCDR_CD_")
    ]

    print("\nICD-9 procedure code patterns:")
    for column in procedure_columns:
        non_missing = df[column].dropna()

        print(
            f"{column:20} | "
            f"Non-missing: {len(non_missing):,} | "
            f"Unique: {non_missing.nunique():,} | "
            f"Sample: {non_missing.astype(str).head(5).tolist()}"
        )

    # HCPCS fields
    hcpcs_columns = [
        column for column in df.columns
        if column.startswith("HCPCS_CD_")
    ]

    print("\nHCPCS code patterns:")
    for column in hcpcs_columns:
        non_missing = df[column].dropna()

        print(
            f"{column:20} | "
            f"Non-missing: {len(non_missing):,} | "
            f"Unique: {non_missing.nunique():,} | "
            f"Sample: {non_missing.astype(str).head(5).tolist()}"
        )


inspect_claim_codes(
    files["inpatient"],
    "INPATIENT CLAIMS"
)

inspect_claim_codes(
    files["outpatient"],
    "OUTPATIENT CLAIMS"
)


INPATIENT CLAIMS

SEGMENT values:
[1, 2]

ICD-9 diagnosis code patterns:
ICD9_DGNS_CD_1       | Non-missing: 66,678 | Unique: 2,740 | Sample: ['7802', '1970', '6186', '29623', '3569']
ICD9_DGNS_CD_2       | Non-missing: 66,247 | Unique: 2,961 | Sample: ['78820', '4019', '2948', '30390', '4019']
ICD9_DGNS_CD_3       | Non-missing: 65,492 | Unique: 2,909 | Sample: ['V4501', '5853', '56400', '71690', '3542']
ICD9_DGNS_CD_4       | Non-missing: 64,005 | Unique: 2,907 | Sample: ['4280', '7843', '34590', 'V8801', '42830']
ICD9_DGNS_CD_5       | Non-missing: 61,639 | Unique: 2,915 | Sample: ['2720', '2768', 'V1581', '78820', '2724']
ICD9_DGNS_CD_6       | Non-missing: 58,369 | Unique: 2,843 | Sample: ['4019', '71590', '32723', '2639', 'V4581']
ICD9_DGNS_CD_7       | Non-missing: 54,407 | Unique: 2,823 | Sample: ['V4502', '2724', '7840', '4019', '78729']
ICD9_DGNS_CD_8       | Non-missing: 49,947 | Unique: 2,696 | Sample: ['73300', '19889', '7856', 'V103', '04189']
ICD9_DGNS_CD_9       | Non-

## Claims Coded Field Findings

The claims coded-field assessment identified several structural patterns.

### Segment Field

Both inpatient and outpatient claims contain only:

- `SEGMENT = 1`
- `SEGMENT = 2`

No unexpected segment values were identified.

### Inpatient Diagnosis Codes

The inpatient dataset contains substantial numbers of ICD-9 diagnosis codes across multiple diagnosis positions.

Later diagnosis positions contain progressively fewer populated records, which is consistent with multiple diagnosis slots.

The observed codes include numeric-looking ICD-9 codes as well as codes beginning with `V` and `E`.

### Inpatient Procedure Codes

The inpatient procedure fields contain relatively fewer populated records than the diagnosis fields.

Some values are represented by pandas as floating-point values, for example `7092.0`, while other observations contain alphanumeric values such as `E8889` and `V5866`.

These fields therefore require careful type handling during cleaning. They should be treated as coded categorical values rather than continuous numeric measurements.

### Inpatient HCPCS Fields

All 45 inpatient HCPCS fields contain zero non-missing values.

This confirms that the HCPCS fields are structurally unavailable in this inpatient extract.

They will not be used for inpatient HCPCS analysis.

### Outpatient Diagnosis Codes

Outpatient diagnosis fields are substantially populated.

The first diagnosis position contains 785,170 non-missing values, with progressively fewer populated records in later diagnosis positions.

This pattern is consistent with multiple diagnosis slots.

### Outpatient Procedure Codes

Outpatient ICD-9 procedure fields are extremely sparse compared with the diagnosis and HCPCS fields.

The number of non-missing records ranges from 200 in the first procedure field to only 29 in the sixth procedure field.

Their analytical usefulness will therefore be evaluated before inclusion in downstream analysis.

### Outpatient HCPCS Codes

Outpatient HCPCS fields are substantially populated.

`HCPCS_CD_1` contains 646,260 non-missing records and 2,533 unique codes.

Multiple subsequent HCPCS fields are also populated and contain a wide range of service codes.

These fields may provide useful information for outpatient service and procedure analysis.

### Cleaning Decision

- `SEGMENT` will be retained as a categorical field.
- Diagnosis, procedure, and HCPCS codes will be treated as categorical/code fields rather than continuous numeric variables.
- Code representations such as `7092.0` will be investigated before cleaning.
- No coded values will be deleted or transformed solely because of their current pandas data type.
- Completely unavailable inpatient HCPCS fields will be excluded from downstream analytical use rather than artificially imputed.

> **Raw data remains unchanged.**

## 6. Financial Value Assessment

Financial fields are central to the claims and revenue analysis.

This assessment evaluates:

- Negative claim payment amounts
- Zero claim payment amounts
- Negative primary payer amounts
- Extreme payment values
- Basic relationships between payment fields

The objective is to identify financial values that require investigation before cleaning.

> **Important:** Negative or zero values will not automatically be treated as errors. Their treatment will be determined after examining their frequency and context.

In [11]:
# --------------------------------------------------
# Financial Value Assessment
# --------------------------------------------------

financial_fields = [
    "CLM_PMT_AMT",
    "NCH_PRMRY_PYR_CLM_PD_AMT"
]


def assess_financial_values(file_path, dataset_name):

    summary = {}

    for column in financial_fields:
        summary[column] = {
            "Records": 0,
            "Negative_Count": 0,
            "Zero_Count": 0,
            "Positive_Count": 0,
            "Minimum": None,
            "Maximum": None
        }

    for chunk in pd.read_csv(
        file_path,
        usecols=financial_fields,
        chunksize=100_000,
        low_memory=False
    ):

        for column in financial_fields:

            series = pd.to_numeric(
                chunk[column],
                errors="coerce"
            ).dropna()

            if len(series) == 0:
                continue

            summary[column]["Records"] += len(series)
            summary[column]["Negative_Count"] += (series < 0).sum()
            summary[column]["Zero_Count"] += (series == 0).sum()
            summary[column]["Positive_Count"] += (series > 0).sum()

            chunk_min = series.min()
            chunk_max = series.max()

            if (
                summary[column]["Minimum"] is None
                or chunk_min < summary[column]["Minimum"]
            ):
                summary[column]["Minimum"] = chunk_min

            if (
                summary[column]["Maximum"] is None
                or chunk_max > summary[column]["Maximum"]
            ):
                summary[column]["Maximum"] = chunk_max

    results = []

    for column, values in summary.items():

        results.append({
            "Dataset": dataset_name,
            "Field": column,
            **values
        })

    return pd.DataFrame(results)


financial_results = []

for name in ["inpatient", "outpatient"]:

    print(f"Assessing financial values: {name}")

    result = assess_financial_values(
        files[name],
        name
    )

    financial_results.append(result)


financial_df = pd.concat(
    financial_results,
    ignore_index=True
)

display(financial_df)

Assessing financial values: inpatient
Assessing financial values: outpatient


,Dataset,Field,Records,Negative_Count,Zero_Count,Positive_Count,Minimum,Maximum
0,inpatient,CLM_PMT_AMT,66773,55,2160,64558,-8000.0,57000.0
1,inpatient,NCH_PRMRY_PYR_CLM_PD_AMT,66773,0,65246,1527,0.0,68000.0
2,outpatient,CLM_PMT_AMT,790790,2566,30105,758119,-100.0,3300.0
3,outpatient,NCH_PRMRY_PYR_CLM_PD_AMT,790790,0,783826,6964,0.0,14000.0


## Financial Value Findings

The financial assessment identified negative, zero, and positive payment values in the claims datasets.

### Inpatient Claims

For 66,773 inpatient records:

- `CLM_PMT_AMT` contains 55 negative values.
- 2,160 records have a zero `CLM_PMT_AMT`.
- 64,558 records have a positive `CLM_PMT_AMT`.
- The observed range is -8,000 to 57,000.
- `NCH_PRMRY_PYR_CLM_PD_AMT` contains no negative values.
- 65,246 records have a zero primary-payer amount.
- 1,527 records have a positive primary-payer amount.
- The observed range is 0 to 68,000.

### Outpatient Claims

For 790,790 outpatient records:

- `CLM_PMT_AMT` contains 2,566 negative values.
- 30,105 records have a zero `CLM_PMT_AMT`.
- 758,119 records have a positive `CLM_PMT_AMT`.
- The observed range is -100 to 3,300.
- `NCH_PRMRY_PYR_CLM_PD_AMT` contains no negative values.
- 783,826 records have a zero primary-payer amount.
- 6,964 records have a positive primary-payer amount.
- The observed range is 0 to 14,000.

### Assessment

Negative values occur only in `CLM_PMT_AMT` and are relatively uncommon compared with the total number of claims.

Zero payment values are substantially more common, particularly in
`NCH_PRMRY_PYR_CLM_PD_AMT`.

These values will not automatically be classified as errors because payment
fields can represent different financial components of a claim.

### Cleaning Decision

- Financial fields will remain numeric.
- Negative `CLM_PMT_AMT` values will be retained initially and investigated further.
- Zero payment values will be retained.
- No financial values will be imputed or deleted at this stage.
- Any transformation of financial values will be documented during the cleaning stage.

> **Important:** The presence of a negative payment value does not by itself establish that the record is invalid. Context and source-field definitions must be considered before making a cleaning decision.

## 7. Cross-Field Consistency Assessment

Cross-field consistency checks whether related variables agree with each other
and follow expected logical relationships.

The assessment will examine:

- Claim start and end dates
- Inpatient utilization days
- Beneficiary-to-claim relationships
- Financial-field relationships

These checks help distinguish missing values from logically inconsistent records.

> **Important:** Records will not be modified or removed during this assessment.

In [12]:
# --------------------------------------------------
# Cross-Field Consistency Assessment
# --------------------------------------------------

# 1. Inpatient utilization validation

inpatient_utilization = []

for chunk in pd.read_csv(
    files["inpatient"],
    usecols=["CLM_UTLZTN_DAY_CNT"],
    chunksize=100_000,
    low_memory=False
):
    series = pd.to_numeric(
        chunk["CLM_UTLZTN_DAY_CNT"],
        errors="coerce"
    )

    inpatient_utilization.append(
        pd.DataFrame({
            "Negative_Days": [(series < 0).sum()],
            "Zero_Days": [(series == 0).sum()],
            "Positive_Days": [(series > 0).sum()],
            "Missing_Days": [series.isna().sum()],
            "Minimum_Days": [series.min()],
            "Maximum_Days": [series.max()]
        })
    )

utilization_df = pd.concat(
    inpatient_utilization,
    ignore_index=True
)

utilization_summary = pd.DataFrame({
    "Metric": [
        "Negative Days",
        "Zero Days",
        "Positive Days",
        "Missing Days",
        "Minimum Days",
        "Maximum Days"
    ],
    "Value": [
        utilization_df["Negative_Days"].sum(),
        utilization_df["Zero_Days"].sum(),
        utilization_df["Positive_Days"].sum(),
        utilization_df["Missing_Days"].sum(),
        utilization_df["Minimum_Days"].min(),
        utilization_df["Maximum_Days"].max()
    ]
})

display(utilization_summary)


# 2. Beneficiary-to-claim relationship validation

beneficiary_ids = set()

for name in [
    "beneficiary_2008",
    "beneficiary_2009",
    "beneficiary_2010"
]:
    for chunk in pd.read_csv(
        files[name],
        usecols=["DESYNPUF_ID"],
        chunksize=100_000,
        low_memory=False
    ):
        beneficiary_ids.update(
            chunk["DESYNPUF_ID"].dropna()
        )

print("\nUnique beneficiary IDs across annual files:")
print(len(beneficiary_ids))


for name in ["inpatient", "outpatient"]:

    matched = 0
    unmatched = 0

    for chunk in pd.read_csv(
        files[name],
        usecols=["DESYNPUF_ID"],
        chunksize=100_000,
        low_memory=False
    ):

        for beneficiary_id in chunk["DESYNPUF_ID"].dropna():

            if beneficiary_id in beneficiary_ids:
                matched += 1
            else:
                unmatched += 1

    print(f"\n{name}")
    print(f"Matched claims:   {matched:,}")
    print(f"Unmatched claims: {unmatched:,}")

,Metric,Value
0,Negative Days,0.0
1,Zero Days,2266.0
2,Positive Days,64439.0
3,Missing Days,68.0
4,Minimum Days,0.0
5,Maximum Days,136.0



Unique beneficiary IDs across annual files:
116352

inpatient
Matched claims:   66,773
Unmatched claims: 0

outpatient
Matched claims:   790,790
Unmatched claims: 0


## Cross-Field Consistency Findings

The cross-field consistency assessment identified no major referential-integrity
or utilization-value inconsistencies.

### Inpatient Utilization

The `CLM_UTLZTN_DAY_CNT` field contains:

- 0 negative values
- 2,266 zero-day records
- 64,439 positive-day records
- 68 missing values
- Minimum: 0 days
- Maximum: 136 days

No negative utilization-day values were identified.

The 68 missing utilization values correspond to records where the field was
not populated and will be handled according to the field's analytical relevance
during cleaning.

### Beneficiary-to-Claim Consistency

There are 116,352 unique beneficiary identifiers across the annual beneficiary
files.

All claims successfully matched a beneficiary identifier:

- Inpatient: 66,773 matched; 0 unmatched
- Outpatient: 790,790 matched; 0 unmatched

This indicates complete referential linkage between the claims datasets and the
available beneficiary population.

### Cleaning Decision

- Negative utilization days will not require correction because none were found.
- Zero utilization days will be retained.
- Missing utilization days will be retained initially and assessed during cleaning.
- No claims will be removed because of unmatched beneficiary identifiers.

> **Overall:** No major cross-field consistency issue was identified that
> requires immediate record-level deletion or correction.

## 8. Code Format and Representation Assessment

Healthcare diagnosis, procedure, and HCPCS codes are categorical identifiers,
not continuous numerical measurements.

During validation, some ICD-9 procedure fields were observed with mixed
representations, including numeric-looking values such as `7092.0` and
alphanumeric values such as `E8889`.

This assessment investigates the raw representation of selected code fields
before defining any cleaning rules.

### Objectives

- Identify mixed data representations
- Check whether numeric-looking codes are stored with decimal notation
- Determine whether alphanumeric codes are present
- Identify potential leading-zero risks
- Define appropriate data types for downstream analysis

> **Important:** No code values will be modified during this assessment.

In [13]:
# --------------------------------------------------
# Code Format and Representation Assessment
# --------------------------------------------------

code_fields = [
    "ICD9_DGNS_CD_1",
    "ICD9_PRCDR_CD_1",
    "HCPCS_CD_1"
]


def inspect_code_representation(file_path, dataset_name):

    print(f"\n{'=' * 70}")
    print(dataset_name)
    print(f"{'=' * 70}")

    df = pd.read_csv(
        file_path,
        usecols=code_fields,
        dtype=str,
        low_memory=False
    )

    for column in code_fields:

        series = df[column].dropna().astype(str).str.strip()

        if series.empty:
            print(f"\n{column}: No non-missing values")
            continue

        numeric_like = series.str.fullmatch(r"\d+(\.0)?").sum()
        alphabetic = series.str.contains(
            r"[A-Za-z]",
            regex=True
        ).sum()

        decimal_values = series.str.contains(
            r"\.0$",
            regex=True
        ).sum()

        print(f"\n{column}")
        print(f"Non-missing values : {len(series):,}")
        print(f"Unique values      : {series.nunique():,}")
        print(f"Numeric-like       : {numeric_like:,}")
        print(f"Contains letters   : {alphabetic:,}")
        print(f"Ends with .0       : {decimal_values:,}")
        print(
            "Sample values      :",
            series.head(10).tolist()
        )


inspect_code_representation(
    files["inpatient"],
    "INPATIENT CLAIMS"
)

inspect_code_representation(
    files["outpatient"],
    "OUTPATIENT CLAIMS"
)


INPATIENT CLAIMS

ICD9_DGNS_CD_1
Non-missing values : 66,678
Unique values      : 2,740
Numeric-like       : 64,521
Contains letters   : 2,157
Ends with .0       : 0
Sample values      : ['7802', '1970', '6186', '29623', '3569', '486', '33811', '49121', 'V5789', '7366']

ICD9_PRCDR_CD_1
Non-missing values : 38,231
Unique values      : 1,294
Numeric-like       : 38,231
Contains letters   : 0
Ends with .0       : 0
Sample values      : ['7092', '8659', '0073', '0050', '9904', '4523', '9671', '7935', '4516', '8622']

HCPCS_CD_1: No non-missing values

OUTPATIENT CLAIMS

ICD9_DGNS_CD_1
Non-missing values : 785,170
Unique values      : 11,133
Numeric-like       : 620,076
Contains letters   : 165,094
Ends with .0       : 0
Sample values      : ['V5841', 'V5832', '9594', '78943', '6009', '6115', '2723', 'V821', '7902', '7232']

ICD9_PRCDR_CD_1
Non-missing values : 200
Unique values      : 99
Numeric-like       : 200
Contains letters   : 0
Ends with .0       : 0
Sample values      : ['3995', 

## Code Format and Representation Findings

The code-format assessment confirms that healthcare diagnosis, procedure, and
HCPCS fields should be treated as categorical string fields.

### Diagnosis Codes

Both inpatient and outpatient diagnosis fields contain:

- Numeric-looking ICD-9 codes
- Alphanumeric ICD-9 codes
- Codes beginning with `V`

For example:

- Inpatient: `7802`, `1970`, `V5789`
- Outpatient: `V5841`, `9594`, `V821`

This confirms that diagnosis codes cannot be reliably represented as continuous
numeric variables.

### Procedure Codes

Inpatient and outpatient ICD-9 procedure codes are predominantly
numeric-looking strings.

Some codes contain leading zeros, such as:

- `0073`
- `0050`
- `0066`

These leading zeros are part of the code representation and should not be
removed during cleaning.

### HCPCS Codes

Outpatient HCPCS fields contain both numeric-looking and alphanumeric codes.

Examples include:

- `85610`
- `36415`
- `G0206`
- `G0202`

Inpatient HCPCS fields contain no populated values in this extract.

### Important Observation

When the fields are explicitly read as strings, no `.0` suffixes are present
in the examined code fields.

The previously observed floating-point appearance of some procedure codes was
therefore related to data-type inference rather than evidence that the source
codes themselves contain decimal values.

### Cleaning Decision

For downstream processing:

- ICD-9 diagnosis codes will be stored as strings.
- ICD-9 procedure codes will be stored as strings.
- HCPCS codes will be stored as strings.
- Leading zeros will be preserved.
- Alphanumeric codes will be preserved.
- Codes will not be converted to numeric data types.
- Code values will not be mathematically transformed.

> **Raw source data remains unchanged.**

## 9. Analytical Relevance Assessment

Data quality is not determined only by whether a field is complete or valid.

A field may be structurally valid but have limited analytical value for the
business questions defined for this project.

This assessment classifies important fields according to their expected
analytical relevance.

### Classification

- **Core:** Required for primary claims, payment, provider, beneficiary, or
  utilization analysis.
- **Useful:** Potentially valuable for segmentation or deeper analysis.
- **Low Priority:** Valid field with limited relevance to the current project.
- **Unavailable:** Field is completely missing or structurally unavailable in
  the source extract.

### Important Principle

Analytical relevance does not mean that non-core fields will be deleted from
the raw dataset.

The classification will be used later to determine which fields are retained
in the cleaned analytical datasets.

> **Raw source data remains unchanged.**

In [14]:
# --------------------------------------------------
# Analytical Relevance Assessment
# --------------------------------------------------

analytical_relevance = pd.DataFrame([
    {
        "Dataset": "Beneficiary",
        "Field": "DESYNPUF_ID",
        "Category": "Core",
        "Purpose": "Unique beneficiary identifier"
    },
    {
        "Dataset": "Beneficiary",
        "Field": "BENE_SEX_IDENT_CD",
        "Category": "Useful",
        "Purpose": "Demographic segmentation"
    },
    {
        "Dataset": "Beneficiary",
        "Field": "BENE_RACE_CD",
        "Category": "Useful",
        "Purpose": "Demographic segmentation"
    },
    {
        "Dataset": "Beneficiary",
        "Field": "BENE_ESRD_IND",
        "Category": "Useful",
        "Purpose": "Clinical risk segmentation"
    },
    {
        "Dataset": "Beneficiary",
        "Field": "BENE_DEATH_DT",
        "Category": "Low Priority",
        "Purpose": "Mortality-related analysis; very high missingness"
    },
    {
        "Dataset": "Inpatient",
        "Field": "CLM_ID",
        "Category": "Core",
        "Purpose": "Claim identification"
    },
    {
        "Dataset": "Inpatient",
        "Field": "SEGMENT",
        "Category": "Core",
        "Purpose": "Claim segment identification"
    },
    {
        "Dataset": "Inpatient",
        "Field": "DESYNPUF_ID",
        "Category": "Core",
        "Purpose": "Beneficiary linkage"
    },
    {
        "Dataset": "Inpatient",
        "Field": "CLM_FROM_DT",
        "Category": "Core",
        "Purpose": "Claim date and trend analysis"
    },
    {
        "Dataset": "Inpatient",
        "Field": "CLM_THRU_DT",
        "Category": "Core",
        "Purpose": "Claim duration analysis"
    },
    {
        "Dataset": "Inpatient",
        "Field": "CLM_PMT_AMT",
        "Category": "Core",
        "Purpose": "Claim payment analysis"
    },
    {
        "Dataset": "Inpatient",
        "Field": "NCH_PRMRY_PYR_CLM_PD_AMT",
        "Category": "Core",
        "Purpose": "Primary payer analysis"
    },
    {
        "Dataset": "Inpatient",
        "Field": "CLM_UTLZTN_DAY_CNT",
        "Category": "Core",
        "Purpose": "Utilization analysis"
    },
    {
        "Dataset": "Inpatient",
        "Field": "PRVDR_NUM",
        "Category": "Core",
        "Purpose": "Provider-level analysis"
    },
    {
        "Dataset": "Inpatient",
        "Field": "ICD9_DGNS_CD_1",
        "Category": "Useful",
        "Purpose": "Diagnosis analysis"
    },
    {
        "Dataset": "Inpatient",
        "Field": "ICD9_PRCDR_CD_1",
        "Category": "Useful",
        "Purpose": "Procedure analysis"
    },
    {
        "Dataset": "Inpatient",
        "Field": "HCPCS_CD_1",
        "Category": "Unavailable",
        "Purpose": "No populated values in this extract"
    },
    {
        "Dataset": "Outpatient",
        "Field": "CLM_ID",
        "Category": "Core",
        "Purpose": "Claim identification"
    },
    {
        "Dataset": "Outpatient",
        "Field": "SEGMENT",
        "Category": "Core",
        "Purpose": "Claim segment identification"
    },
    {
        "Dataset": "Outpatient",
        "Field": "DESYNPUF_ID",
        "Category": "Core",
        "Purpose": "Beneficiary linkage"
    },
    {
        "Dataset": "Outpatient",
        "Field": "CLM_FROM_DT",
        "Category": "Core",
        "Purpose": "Claim date and trend analysis"
    },
    {
        "Dataset": "Outpatient",
        "Field": "CLM_THRU_DT",
        "Category": "Core",
        "Purpose": "Claim duration analysis"
    },
    {
        "Dataset": "Outpatient",
        "Field": "CLM_PMT_AMT",
        "Category": "Core",
        "Purpose": "Claim payment analysis"
    },
    {
        "Dataset": "Outpatient",
        "Field": "NCH_PRMRY_PYR_CLM_PD_AMT",
        "Category": "Core",
        "Purpose": "Primary payer analysis"
    },
    {
        "Dataset": "Outpatient",
        "Field": "PRVDR_NUM",
        "Category": "Core",
        "Purpose": "Provider-level analysis"
    },
    {
        "Dataset": "Outpatient",
        "Field": "ICD9_DGNS_CD_1",
        "Category": "Useful",
        "Purpose": "Diagnosis analysis"
    },
    {
        "Dataset": "Outpatient",
        "Field": "ICD9_PRCDR_CD_1",
        "Category": "Low Priority",
        "Purpose": "Very sparse procedure field"
    },
    {
        "Dataset": "Outpatient",
        "Field": "HCPCS_CD_1",
        "Category": "Useful",
        "Purpose": "Outpatient service/procedure analysis"
    }
])

display(analytical_relevance)

,Dataset,Field,Category,Purpose
0,Beneficiary,DESYNPUF_ID,Core,Unique beneficiary identifier
1,Beneficiary,BENE_SEX_IDENT_CD,Useful,Demographic segmentation
2,Beneficiary,BENE_RACE_CD,Useful,Demographic segmentation
3,Beneficiary,BENE_ESRD_IND,Useful,Clinical risk segmentation
4,Beneficiary,BENE_DEATH_DT,Low Priority,Mortality-related analysis; very high missingness
5,Inpatient,CLM_ID,Core,Claim identification
6,Inpatient,SEGMENT,Core,Claim segment identification
7,Inpatient,DESYNPUF_ID,Core,Beneficiary linkage
8,Inpatient,CLM_FROM_DT,Core,Claim date and trend analysis
9,Inpatient,CLM_THRU_DT,Core,Claim duration analysis


## Analytical Relevance Findings

The analytical relevance assessment identified the fields most closely aligned
with the project's claims, payment, provider, beneficiary, and utilization
objectives.

### Core Fields

Core fields include:

- Beneficiary identifier
- Claim identifier
- Claim segment
- Claim dates
- Claim payment amount
- Primary payer payment amount
- Inpatient utilization days
- Provider identifier

These fields form the foundation of the downstream claims and revenue analysis.

### Useful Fields

Several coded and demographic fields provide additional analytical value,
including:

- Beneficiary demographic indicators
- Selected clinical indicators
- Diagnosis codes
- Inpatient procedure codes
- Outpatient HCPCS codes

These fields can support segmentation and deeper utilization analysis.

### Low-Priority Fields

Some fields may be structurally valid but have limited value for the current
business questions.

For example:

- `BENE_DEATH_DT` has very high missingness.
- Outpatient ICD-9 procedure fields are extremely sparse.

These fields will not automatically be deleted from the raw data. Their
inclusion in the cleaned analytical dataset will depend on downstream
analytical requirements.

### Unavailable Fields

The inpatient HCPCS fields contain no populated values in this extract.

These fields will not be artificially populated or imputed.

### Overall Cleaning Principle

The quality assessment indicates that cleaning should focus on:

1. Correct data types
2. Date standardization
3. Code preservation
4. Missing-value handling
5. Financial-value investigation
6. Removal of structurally unusable fields from analytical datasets where appropriate

The raw source files will remain unchanged.

> **The purpose of data cleaning is to create analysis-ready datasets while
> preserving the integrity of the original source data.**

## 10. Data Quality Assessment — Final Summary

The data quality assessment identified the following major findings:

| Data Quality Dimension | Finding | Cleaning Implication |
|---|---|---|
| Completeness | Missingness concentrated in specific fields | Assess fields individually |
| Business-key uniqueness | No duplicate business keys | No duplicate-key removal required |
| Exact duplicates | No exact duplicate rows identified | No exact-duplicate removal required |
| Coded-field validity | Selected coded fields show consistent patterns | Preserve code values |
| Financial values | Negative and zero payments are present | Investigate; do not automatically delete |
| Cross-field consistency | No unmatched beneficiaries or negative utilization days | No immediate record deletion required |
| Code representation | Codes require string treatment | Preserve leading zeros and alphanumeric values |
| Analytical relevance | Core, useful, low-priority, and unavailable fields identified | Define cleaned analytical datasets accordingly |

### Overall Assessment

The source datasets are structurally suitable for healthcare claims and
revenue-oriented analysis, subject to documented cleaning and transformation.

The most important cleaning considerations are:

- Preserve healthcare codes as strings.
- Preserve meaningful leading zeros.
- Standardize date fields.
- Handle missing values according to field purpose.
- Investigate negative claim payment values rather than automatically removing them.
- Retain zero payment values unless a documented analytical rule requires otherwise.
- Exclude completely unavailable fields from downstream analysis rather than imputing them.
- Maintain beneficiary-to-claim referential integrity.
- Avoid treating synthetic claims data as representative of real-world Medicare utilization or payment behavior.

No raw source records will be modified as part of this quality assessment.

### Next Stage

The next notebook, `03_Data_Cleaning.ipynb`, will implement the documented
cleaning rules and create analysis-ready datasets in `02_Cleaned_Data/`.